In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv('../dados/dataset_fullstack_900_smarttv_privacidade.csv')
df_dicionario = pd.read_csv('../dados/dicionario_fullstack_smarttv.csv')    

In [9]:
# 1. Duplicatas
df = df.drop_duplicates(subset=['id_registro'])

# 2. Padronizar categorias: .str.strip() ANTES do mapeamento resolve o espaço invisível
df['categoria_conteudo'] = df['categoria_conteudo'].str.strip()
map_conteudo = {'FILMES': 'Filmes', 'séries': 'Series', 'ESPOrtes': 'Esportes', 'noticias': 'Noticias', 'infantil': 'Infantil'}
df['categoria_conteudo'] = df['categoria_conteudo'].replace(map_conteudo)

df['estado_microfone'] = df['estado_microfone'].str.strip()
map_mic = {'ativo': 'Ativo', 'INATIVO': 'Inativo'}
df['estado_microfone'] = df['estado_microfone'].replace(map_mic)

df['categoria_telemetria'] = df['categoria_telemetria'].str.strip()
map_telemetria = {'acr/conteudo': 'ACR/conteudo', 'publicidade': 'Publicidade', 'TELEMETRIA SISTEMA': 'Telemetria sistema', 'Metadados de Eventos': 'Metadados de eventos'}
df['categoria_telemetria'] = df['categoria_telemetria'].replace(map_telemetria)

# 3. data_hora
df['data_hora'] = pd.to_datetime(df['data_hora'], errors='coerce')

# 4. Valores inválidos -> NaN
df.loc[(df['minutos_visualizados'] < 0) | (df['minutos_visualizados'] > 10), 'minutos_visualizados'] = np.nan
df.loc[(df['confianca_classificacao'] < 0) | (df['confianca_classificacao'] > 1), 'confianca_classificacao'] = np.nan
df.loc[df['trafego_saida_kb'] < 0, 'trafego_saida_kb'] = np.nan
df.loc[df['anuncios_exibidos'] < 0, 'anuncios_exibidos'] = np.nan

# 5. Interações > anúncios (Ambos viram NaN)
mask_int = df['interacoes_anuncios'] > df['anuncios_exibidos']
df.loc[mask_int, ['interacoes_anuncios', 'anuncios_exibidos']] = np.nan

# 6. Incoerências do microfone 
mask_inativo_erro = (df['estado_microfone'] == 'Inativo') & (df['origem_microfone'] != 'Nao se aplica')
df.loc[mask_inativo_erro, 'origem_microfone'] = 'Nao se aplica'
df.loc[mask_inativo_erro, 'classe_som'] = 'Nao se aplica'

mask_ativo_erro = (df['estado_microfone'] == 'Ativo') & (df['origem_microfone'] == 'Nao se aplica')
df.loc[mask_ativo_erro, 'origem_microfone'] = np.nan 

# 7. Ausentes (Mantemos NaN em confiança e criamos flag para a "ausência legítima")
df['consentimento_coleta'] = df['consentimento_coleta'].fillna('Indeterminado')

mask_legitimo = (df['estado_microfone'] == 'Inativo') | (df['classe_som'] == 'Silencio')
df['ausencia_confianca_legitima'] = df['confianca_classificacao'].isna() & mask_legitimo

# 8. Outliers de Tráfego
Q1 = df['trafego_saida_kb'].quantile(0.25)
Q3 = df['trafego_saida_kb'].quantile(0.75)
IQR = Q3 - Q1
limite_sup = Q3 + 1.5 * IQR
df['is_outlier_trafego'] = df['trafego_saida_kb'] > limite_sup

# Novas Variáveis Derivadas
df['hora_dia'] = df['data_hora'].dt.hour

# Taxa de interação: agora retorna NaN se anúncios exibidos for 0 ou NaN
df['taxa_interacao'] = np.where(df['anuncios_exibidos'] > 0, 
                                df['interacoes_anuncios'] / df['anuncios_exibidos'], 
                                np.nan)

In [8]:
# Passo 1: Duplicatas (Feito anteriormente, mas necessário para a base limpa)
df = df.drop_duplicates(subset=['id_registro'])

# Passo 2: Padronizar categorias (Mapeamento explícito para não quebrar regras)
map_conteudo = {' FILMES': 'Filmes', 'séries': 'Series', 'ESPOrtes': 'Esportes', 'noticias': 'Noticias', ' infantil': 'Infantil'}
df['categoria_conteudo'] = df['categoria_conteudo'].replace(map_conteudo)

map_mic = {'ativo': 'Ativo', ' Inativo': 'Inativo', 'INATIVO': 'Inativo'}
df['estado_microfone'] = df['estado_microfone'].replace(map_mic)

map_telemetria = {'acr/conteudo': 'ACR/conteudo', ' publicidade': 'Publicidade', 'TELEMETRIA SISTEMA': 'Telemetria sistema', 'Metadados de Eventos': 'Metadados de eventos'}
df['categoria_telemetria'] = df['categoria_telemetria'].replace(map_telemetria)

# Passo 3: data_hora -> datetime
df['data_hora'] = pd.to_datetime(df['data_hora'], errors='coerce')
nat_count = df['data_hora'].isna().sum()

# Passo 4: Valores inválidos -> NaN
df.loc[(df['minutos_visualizados'] < 0) | (df['minutos_visualizados'] > 10), 'minutos_visualizados'] = np.nan
df.loc[(df['confianca_classificacao'] < 0) | (df['confianca_classificacao'] > 1), 'confianca_classificacao'] = np.nan
df.loc[df['trafego_saida_kb'] < 0, 'trafego_saida_kb'] = np.nan
df.loc[df['anuncios_exibidos'] < 0, 'anuncios_exibidos'] = np.nan

# Passo 5: Interações > anúncios
# Abordagem: Transformar ambas as métricas daquela linha em NaN, pois não sabemos qual campo falhou.
mask_int = df['interacoes_anuncios'] > df['anuncios_exibidos']
df.loc[mask_int, ['interacoes_anuncios', 'anuncios_exibidos']] = np.nan

# Passo 6: Incoerências do microfone (Caminho Duplo)
mask_inativo_erro = (df['estado_microfone'] == 'Inativo') & (df['origem_microfone'] != 'Nao se aplica')
df.loc[mask_inativo_erro, 'origem_microfone'] = 'Nao se aplica'
df.loc[mask_inativo_erro, 'classe_som'] = 'Nao se aplica'

mask_ativo_erro = (df['estado_microfone'] == 'Ativo') & (df['origem_microfone'] == 'Nao se aplica')
df.loc[mask_ativo_erro, 'origem_microfone'] = np.nan # Vira nulo, pois não sabemos qual a verdadeira origem

# Passo 7: Ausentes
df['consentimento_coleta'] = df['consentimento_coleta'].fillna('Indeterminado')

# Confiança vazia com mic inativo (ou silêncio) é legítima (0.0). O resto continua NaN (ausência suspeita).
mask_legitimo = (df['estado_microfone'] == 'Inativo') | (df['classe_som'] == 'Silencio')
df.loc[df['confianca_classificacao'].isna() & mask_legitimo, 'confianca_classificacao'] = 0.0

# Passo 8 & Novas Variáveis
# Nova Variável 1: Flag de Outlier de Tráfego
Q1 = df['trafego_saida_kb'].quantile(0.25)
Q3 = df['trafego_saida_kb'].quantile(0.75)
IQR = Q3 - Q1
limite_sup = Q3 + 1.5 * IQR
df['is_outlier_trafego'] = df['trafego_saida_kb'] > limite_sup

# Nova Variável 2: Hora do Dia (extraída de data_hora)
df['hora_dia'] = df['data_hora'].dt.hour

# Nova Variável 3: Taxa de Interação com Anúncios (CTR)
df['taxa_interacao'] = np.where(df['anuncios_exibidos'] > 0, 
                                df['interacoes_anuncios'] / df['anuncios_exibidos'], 
                                0.0)

# Resumo
print(f"1. Duplicatas id_registro removidas. Linhas restantes: {len(df)}")
print(f"3. NaT em data_hora gerados: {nat_count}")
print(f"4. Valores inválidos passados para NaN.")
print(f"5. Interações > anúncios: {mask_int.sum()} linhas tiveram ambas as colunas marcadas como NaN.")
print(f"6. Mic Inativo corrigido para Nao se aplica: {mask_inativo_erro.sum()}. Mic Ativo incorreto passado para NaN: {mask_ativo_erro.sum()}.")
print(f"7. Consentimentos vazios -> Indeterminado. Ausentes legítimos de confiança -> 0.0. Minutos vazios originais + inválidos mantidos como NaN: {df['minutos_visualizados'].isna().sum()}")
print(f"8. Outliers de tráfego flagrados (sem apagar): {df['is_outlier_trafego'].sum()}")
print(f"Novas Variáveis Criadas: 'is_outlier_trafego', 'hora_dia' e 'taxa_interacao'")

1. Duplicatas id_registro removidas. Linhas restantes: 882
3. NaT em data_hora gerados: 0
4. Valores inválidos passados para NaN.
5. Interações > anúncios: 6 linhas tiveram ambas as colunas marcadas como NaN.
6. Mic Inativo corrigido para Nao se aplica: 2. Mic Ativo incorreto passado para NaN: 5.
7. Consentimentos vazios -> Indeterminado. Ausentes legítimos de confiança -> 0.0. Minutos vazios originais + inválidos mantidos como NaN: 30
8. Outliers de tráfego flagrados (sem apagar): 34
Novas Variáveis Criadas: 'is_outlier_trafego', 'hora_dia' e 'taxa_interacao'
